In [1]:
from torch_geometric.datasets import KarateClub
import yaml

dataset = KarateClub()
print(f'Dataset: {dataset}:')
print('======================')
print(f'Number of graphs: {len(dataset)}')
print(f'Number of features: {dataset.num_features}')
print(f'Number of classes: {dataset.num_classes}')

data = dataset[0]  # Get the first graph object.

c:\Users\KETI\anaconda3\envs\ai_tutorial\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset: KarateClub():
Number of graphs: 1
Number of features: 34
Number of classes: 4


In [2]:
# Load best config from wandb sweep
path = "./wandb/sweep-df5mbgsm/config-cqvha3fh.yaml" # put best config path here
with open(path, "r", encoding="utf-8") as f:
    raw_config = yaml.safe_load(f)

# W&B YAML stores each parameter as {value: ...}.
best_config = {
    key: entry["value"]
    for key, entry in raw_config.items()
    if isinstance(entry, dict) and "value" in entry
}

def config_kwargs(prefix):
    return {
        key[len(prefix):]: value # round(value, 3)
        for key, value in best_config.items()
        if key.startswith(prefix)
    }

In [3]:
best_config

{'models.GCN.dropout': 0.2587329053794833,
 'models.GCN.hidden_channels': 32,
 'models.GCN.num_layers': 3,
 'optimizer.Adam.lr': 0.002996199558174318,
 'optimizer.Adam.weight_decay': 5.962101270122169e-06,
 'training.gradient_clip_val': 0.8604019868445902,
 'wandb_project': 'karate-club-gcn-sweep'}

In [4]:
import torch
import torch.nn.functional as F

from utils.model import GCN

import random

def seed_everything(seed):
    """ Set random seeds for reproducibility.
    Args:
        seed (int): Random seed.
    """
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(1234)
model_kwargs = config_kwargs("models.GCN.")
model = GCN(in_channels=dataset.num_features,
            num_classes=dataset.num_classes,
            **model_kwargs)
print("Model config:", model_kwargs)
print(model)

Model config: {'dropout': 0.2587329053794833, 'hidden_channels': 32, 'num_layers': 3}
GCN(
  (convs): ModuleList(
    (0): GCNConv(34, 32)
    (1-2): 2 x GCNConv(32, 32)
  )
  (classifier): Linear(in_features=32, out_features=4, bias=True)
)


In [5]:
criterion = torch.nn.CrossEntropyLoss()  # Define loss criterion.
optimizer_kwargs = config_kwargs("optimizer.Adam.")
# PyYAML may parse scientific notation as a string.
optimizer_kwargs = {key: float(value) for key, value in optimizer_kwargs.items()}
optimizer = torch.optim.Adam(model.parameters(), **optimizer_kwargs)
gradient_clip_val = round(float(best_config["training.gradient_clip_val"]),3)
print("Adam config:", optimizer_kwargs)
print("Gradient clip:", gradient_clip_val)

def train(data):
    model.train()
    optimizer.zero_grad()  # Clear gradients.
    out = model(data.x, data.edge_index)  # Perform a single forward pass.
    loss = criterion(out[data.train_mask], data.y[data.train_mask])  # Compute the loss solely based on the training nodes.
    loss.backward()  # Derive gradients.
    if gradient_clip_val > 0:
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=gradient_clip_val)
    optimizer.step()  # Update parameters based on gradients.
    return loss


for epoch in range(500):
    loss = train(data)
    if epoch % 10 == 0:
        print(f"epoch{epoch}, loss={loss.item():.4f}")

Adam config: {'lr': 0.002996199558174318, 'weight_decay': 5.962101270122169e-06}
Gradient clip: 0.86
epoch0, loss=1.3894
epoch10, loss=1.3575
epoch20, loss=1.2877
epoch30, loss=1.1752
epoch40, loss=0.8655
epoch50, loss=0.4834
epoch60, loss=0.3581
epoch70, loss=0.5354
epoch80, loss=0.1683
epoch90, loss=0.1339
epoch100, loss=0.3259
epoch110, loss=0.0694
epoch120, loss=0.0175
epoch130, loss=0.0726
epoch140, loss=0.0924
epoch150, loss=0.0475
epoch160, loss=0.0080
epoch170, loss=0.0036
epoch180, loss=0.0787
epoch190, loss=0.0642
epoch200, loss=0.0082
epoch210, loss=0.0048
epoch220, loss=0.3322
epoch230, loss=0.0033
epoch240, loss=0.0004
epoch250, loss=0.0050
epoch260, loss=0.0001
epoch270, loss=0.0062
epoch280, loss=0.0013
epoch290, loss=0.7021
epoch300, loss=0.0056
epoch310, loss=0.0012
epoch320, loss=0.0002
epoch330, loss=0.0001
epoch340, loss=0.0007
epoch350, loss=0.0052
epoch360, loss=0.0007
epoch370, loss=0.0002
epoch380, loss=0.0004
epoch390, loss=0.0006
epoch400, loss=0.0024
epoch410

In [6]:
model.eval()
logits = model(data.x, data.edge_index)
test_mask = ~data.train_mask
test_loss = F.cross_entropy(logits[test_mask], data.y[test_mask]).item()
test_accuracy = (
        F.softmax(logits[test_mask],dim=1).argmax(dim=1) == data.y[test_mask]
    ).float().mean().item()

print("test loss: ", test_loss)
print("test accuracy: ", test_accuracy)

test loss:  1.0788182020187378
test accuracy:  0.7333333492279053


In [7]:
# 예측 값
F.softmax(logits[test_mask],dim=1).argmax(dim=1)

tensor([0, 0, 1, 3, 3, 1, 0, 3, 1, 1, 0, 0, 0, 3, 1, 0, 0, 0, 1, 0, 2, 2, 0, 2,
        0, 0, 0, 2, 0, 0])

In [8]:
# 실제 값
data.y[test_mask]

tensor([1, 1, 1, 3, 3, 1, 1, 3, 1, 1, 1, 0, 0, 3, 1, 0, 1, 0, 1, 0, 0, 2, 0, 0,
        2, 0, 0, 2, 0, 0])